In [ ]:
import openai
from openai import OpenAI
import tiktoken
import logging
import pandas as pd
import os
import time
from tqdm import tqdm
import shutil

In [46]:
DEFAULT_SYSTEM_MESSAGE = """You are a scientific visualization expert and prompt engineer.

Create a prompt for an AI image generator that will produce an accurate and visually compelling cover image for a scientific paper.

CRITICAL RULES:
- Promts size should be 50-150 tokens
- Scientific accuracy: Visual elements must correctly represent the scientific concept
- No fictional or decorative elements that contradict the research
- Use correct scientific terminology in the prompt
- Output ONLY the prompt, no extra text
- Image must be in A4 format (portrait orientation, 1:sqrt(2) aspect ratio). Add this information to prompt.

PROMPT COMPONENTS:
1. Core scientific concept (what is being shown)
2. Specific visual details (colors, materials, lighting)
3. Style appropriate for the journal/conference
4. Composition and perspective

EXAMPLES:

Article: "Graphene-Based Battery with 10x Capacity"
Prompt: "Cross-section of graphene layered anode material, lithium ions intercalating between graphene sheets, electron flow visualized as golden energy streams, blue and gray color palette, scientific illustration style, cutaway view, detailed material texture"

Article: "Neural Network Explains Visual Cortex Activity"
Prompt: "Artificial neural network architecture overlaid on primate visual cortex diagram, activation patterns shown as colored heatmaps, connections between nodes mimicking biological pathways, academic illustration style, split-view showing both AI and biology, cool blue to warm red gradient
"""

In [47]:
encoding = tiktoken.get_encoding("cl100k_base")
tokens = encoding.encode(DEFAULT_SYSTEM_MESSAGE)
len(tokens)

294

In [55]:
class Teacher:
    def __init__(self, api_key, model='deepseek-chat', temperature=0.7, max_tokens=150, n=3, in_cost=0.28, out_cost=0.42, system_message=DEFAULT_SYSTEM_MESSAGE, debug=False):
        self.client = OpenAI(api_key=api_key, base_url='https://api.deepseek.com')
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.n = n

        self.result = pd.DataFrame(columns=['id'] + [f'prompt#{i}' for i in range(self.n)])
        self.processed = 0

        self.in_tokens = 0
        self.out_tokens = 0
        self.total_cost = 0
        self.in_cost = in_cost
        self.out_cost = out_cost

        self.tiktoken_encoder = tiktoken.get_encoding('cl100k_base')
        
        self.system_message = {
            "role" : "system", 
            "content" : system_message
        }

        os.makedirs("../logs", exist_ok=True)
        self.logger = logging.getLogger(f"{__name__}.Teacher")
        self.logger.setLevel(logging.DEBUG if debug else logging.INFO)
        if self.logger.handlers:
            self.logger.handlers.clear()
        formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
        file_handler = logging.FileHandler("../logs/teacher.log", encoding='utf-8')
        file_handler.setLevel(logging.DEBUG if debug else logging.INFO)
        file_handler.setFormatter(formatter)
        self.logger.addHandler(file_handler)
        self.logger.propagate = False

        self.logger.info("======================= New session =======================")
        self.logger.debug("Debug mode is ON")
        self.logger.info(f"Teacher initialized with model=\'{model}\', temp={temperature}, n={n}")

    def _call_api(self, messages, max_retries=5):
        base_delay = 15
        for attempt in range(max_retries):
            try:
                response = self.client.chat.completions.create(
                    model = self.model,
                    messages=messages,
                    temperature=self.temperature,
                    max_tokens=self.max_tokens,
                    n=self.n
                )
                self.logger.info(f'Response received, choices={len(response.choices)}')
                return response
            except openai.RateLimitError as e:
                wait = base_delay * (2 ** attempt)
                self.logger.warning(f"Rate Limit reached. Waiting {wait} sec. Attempt: {attempt + 1}/{max_retries}")
                time.sleep(wait)
            except Exception as e:
                self.logger.error(f"An error accured during calling API: {e}")
                raise e

    def _prepare_messages(self, title, abstract, id):
       user_content = f"Generate a prompt for:\nTitle: {title}\n\nAbstract: {abstract}" 
       messages = [self.system_message, {"role" : "user", "content" : user_content}]
       self.logger.debug(f"Prepared messages for {id}. System length: {len(self.system_message['content'])}, user length: {len(user_content)}")
       return messages
    
    def _update_stats(self, response):
        try:
            in_t = response.usage.prompt_tokens
            out_t = response.usage.completion_tokens
        except Exception as e:
            self.logger.error(f"An error occured during stats updating: {e}")
            raise

        self.in_tokens += in_t
        self.out_tokens += out_t

        cost = (in_t * self.in_cost + out_t * self.out_cost) / 1_000_000
        self.total_cost += cost

        self.logger.debug(f"Stats. Curr: {in_t} + {out_t} = {in_t + out_t} ({cost}), total_in_tokens: {self.in_tokens}, total_out_tokens: {self.out_tokens}, total_cost: {self.total_cost}")

    def generate(self, article):
        self.logger.info(f"Start generating prompts for {article['id']}")
        self.logger.debug(f"Article: id={article['id']}, title_len={len(article['title'])}, abstract_len={len(article['abstract'])}")

        try:
            messages = self._prepare_messages(article['title'], article['abstract'], article['id'])
            response = self._call_api(messages)
            self._update_stats(response)
            new_row = {'id' : [article['id']]}
            for i, choice in enumerate(response.choices):
                new_row[f'prompt#{i}'] = choice.message.content
            row = pd.DataFrame(new_row)
            self.result = pd.concat([self.result, row], ignore_index=True)
        except Exception as e:
            self.logger.error(f'An error occured during response: {e}')
            raise

    def generate_from_dataset(self, dataset, start_index=0, save=True, save_dir="../dataset", save_mode='w', checkpoint_dir=None, checkpoint_every=100):
        os.makedirs(checkpoint_dir, exist_ok=True)
        checkpoint_path = checkpoint_dir + "/teacher_check.tsv"
        self.logger.info(f'Starting processing articles from index {start_index}')
        total = dataset.shape[0]
        for i in tqdm(range(start_index, total)):
            article = dataset.iloc[i]
            self.logger.info(f"Processing {i+1}/{total}: {article['id']}")
            try:
                self.generate(article)
                if checkpoint_dir and (i + 1) % checkpoint_every == 0:
                    self._save_checkpoint(checkpoint_path, start_index + self.processed)
            except Exception as e:
                self.logger.error(f"Failed to process {article['id']}: {e}")
                new_row = pd.DataFrame({'id' : [article['id']]})
                for k in range(self.n):
                    new_row[f'prompt#{k}'] = pd.NA
                self.result = pd.concat([self.result, new_row], ignore_index=True)
                continue
        if checkpoint_dir:
            self._save_checkpoint(checkpoint_path, start_index + self.processed)
            self.logger.info(f"Saved final checkpoint.")
        self.logger.info(f"Completed. Processed {self.result.shape[0]} / {dataset.shape[0]} articles")
        rows_with_nan = self.result.isna().any(axis=1).sum()
        if rows_with_nan != 0:
            self.logger.warning(f'Rows with nan: {rows_with_nan}')
        
        if save:
            os.makedirs(save_dir, exist_ok=True)
            source_file = checkpoint_path
            os.rename(checkpoint_path, save_dir + "/prompts.tsv")
            self.logger.info(f"Saved dataset to {save_dir + "prompts.tsv"}")

        return self.result
    
    def _save_checkpoint(self, path, start_idx):
        size = self.result.shape[0]
        try:
            self.result.to_csv(path, sep="\t", mode='w' if start_idx == 0 else 'a', header=True if start_idx == 0 else False, index=False, encoding='utf-8-sig')
        except Exception as e:
            self.logger.error(f"An error occured during saving checkpoint: {e}")
            raise
        self.processed += size
        self.result = self.result[0:0]
        self.logger.info(f"Saved checkpoint for {self.result.shape[0]} articles ({start_idx}:{start_idx + size})")
        self.logger.debug(f"Current size of result: {self.processed}")

In [ ]:
teacher = Teacher(
    api_key='',
    model='deepseek-chat',
    temperature=0.7,
    max_tokens=150,
    n=1,
    system_message=DEFAULT_SYSTEM_MESSAGE,
    debug=True
)
data = pd.read_csv("../dataset/clean.tsv", sep='\t')
teacher.generate_from_dataset(
    dataset=data,
    start_index=0,
    save=True,
    save_dir="../dataset",
    save_mode="w",
    checkpoint_dir="../checkpoint",
    checkpoint_every=1
)

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [01:09<00:00,  6.94s/it]


,id,prompt#0


In [50]:
PROMPT = "Architectural diagram of a Dual Recurrent Attention Unit for Visual Question Answering, showing a neural network with two distinct recurrent pathways processing a question text stream and an image feature map, connected by bidirectional attention flow visualized as glowing, pulsating data streams, with a final joint embedding space, clean and precise 3D schematic style with a white background, using a color palette of deep blue for the visual pathway, amber for the textual pathway, and violet for the attention connections, overhead isometric perspective showing the full data flow from input to output, A4 portrait format, 1:sqrt(2) aspect ratio."
encoding = tiktoken.get_encoding("cl100k_base")
tokens = encoding.encode(PROMPT)
len(tokens)

127